# Day 1-02｜Roboflow Court Keypoint Dataset 與 YOLO Pose 微調
> Python 籃球運動資料分析課程

## 單元目標
- 確認球場 keypoint dataset 的資料夾格式與標註欄位。
- 必要時將 Roboflow COCO keypoint export 轉為 Ultralytics YOLO pose dataset。
- 依需求選擇以基礎權重或課程權重初始化，執行 YOLO pose 微調。
- 使用課程提供模型對參考影片輸出 court keypoint 推論 demo。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 資料位置
- Roboflow COCO keypoint export：`assets/datasets/roboflow_court_coco/`
- 轉換後 YOLO pose dataset：`assets/datasets/roboflow_court_yolo_pose/`
- 課程提供 court keypoint 權重：`assets/models/court_keypoints/yolo26n_basketball_court_pose_best.pt`
- 影片 demo 來源：`assets/raw/reference_videos/`

## 操作流程
1. 檢查資料集與模型權重是否到位。
2. 若是學生自己的 Roboflow 專案，先在網頁完成標註。
3. 到 Roboflow 專案的 `Versions` 頁按 `Generate New Version` / `Publish`。
4. 把新的 `workspace slug`、`project slug`、`version number` 填回本 notebook。
5. 視需要執行 COCO-to-YOLO Pose 轉換。
6. 以 `TRAIN_MODE` 控制是否做微調，以及初始化方式。
7. 使用模型對參考影片輸出 court keypoint demo video。

## 教學用未標註樣本
若本課使用示範專案 `basketball-court-detection-2-85wob`（https://universe.roboflow.com/roboflow-jvuqo/basketball-court-detection-2），其中有 5 張 `train` 圖片會刻意保留為未標註，並加上 `needs-annotation` tag，讓學生 fork 後練習補標。可以直接在 Roboflow Dataset 頁用檔名或 tag 搜尋： 

- `boston-celtics-new-york-knicks-game-1-q1-01_54-01_48_mp4-0000.jpg`
- `boston-celtics-new-york-knicks-game-1-q1-01_54-01_48_mp4-0001.jpg`
- `boston-celtics-new-york-knicks-game-1-q1-01_54-01_48_mp4-0002.jpg`
- `boston-celtics-new-york-knicks-game-1-q1-01_54-01_48_mp4-0003.jpg`
- `boston-celtics-new-york-knicks-game-1-q1-01_54-01_48_mp4-0004.jpg`



In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(preferred=COURSE_ROOT_HINT)


## Roboflow Keypoint Export 格式
`_annotations.coco.json` 需包含 `categories[].keypoints`、`categories[].skeleton`，以及每筆 annotation 的 `keypoints` 欄位。

```text
assets/datasets/roboflow_court_coco/
├── train/
│   ├── _annotations.coco.json
│   └── *.jpg
├── valid/
│   ├── _annotations.coco.json
│   └── *.jpg
└── test/
    ├── _annotations.coco.json
    └── *.jpg
```

轉換完成後，YOLO pose dataset 會放在 `assets/datasets/roboflow_court_yolo_pose/`，其中 `data.yaml` 會帶入 `kpt_shape`、`flip_idx`、`keypoint_names` 與 `keypoint_skeleton`。

若學生是在 Roboflow 網頁上補標註，請先手動發布新的 dataset version；只有新 version 才能被下面的 API 下載流程抓到。


In [ ]:
from src.roboflow_utils import dataset_status

COURT_COCO_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_coco"
COURT_YOLO_POSE_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_yolo_pose"
COURT_MODEL_PATH = (
    COURSE_ROOT
    / "assets"
    / "models"
    / "court_keypoints"
    / "yolo26n_basketball_court_pose_best.pt"
)

print("COCO keypoint export:")
print(dataset_status(COURT_COCO_DIR))
print("\nYOLO pose dataset:")
print(dataset_status(COURT_YOLO_POSE_DIR))
print("\nmodel exists:", COURT_MODEL_PATH.exists(), COURT_MODEL_PATH)


In [ ]:
from getpass import getpass
from src.roboflow_utils import (
    convert_roboflow_coco_keypoints_to_yolo_pose,
    ensure_roboflow_court_pose_dataset,
    find_coco_annotation,
)

# 學生作業建議流程：
# 1. 在 Roboflow 網頁完成 keypoint 標註。
# 2. 到 Versions 頁面按 Generate New Version / Publish。
# 3. 把新的 workspace / project / version 填在下面。
# 4. 將 USE_ROBOFLOW_DOWNLOAD 改成 True。
USE_ROBOFLOW_DOWNLOAD = False
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"
ROBOFLOW_PROJECT = "YOUR_PROJECT"
ROBOFLOW_VERSION = 1
ROBOFLOW_API_KEY = ""
FORCE_DOWNLOAD = False
FORCE_CONVERSION = False

data_yaml = COURT_YOLO_POSE_DIR / "data.yaml"
if USE_ROBOFLOW_DOWNLOAD:
    api_key = ROBOFLOW_API_KEY or getpass("Roboflow API key: ")
    data_yaml = ensure_roboflow_court_pose_dataset(
        coco_dir=COURT_COCO_DIR,
        yolo_pose_dir=COURT_YOLO_POSE_DIR,
        workspace=ROBOFLOW_WORKSPACE,
        project=ROBOFLOW_PROJECT,
        version=ROBOFLOW_VERSION,
        api_key=api_key,
        export_format="coco",
        overwrite_download=FORCE_DOWNLOAD,
        overwrite_conversion=FORCE_CONVERSION,
    )
    print("ready data.yaml:", data_yaml)
elif data_yaml.exists() and not FORCE_CONVERSION:
    print("YOLO pose dataset already exists:", data_yaml)
elif find_coco_annotation(COURT_COCO_DIR / "train") is not None:
    data_yaml = convert_roboflow_coco_keypoints_to_yolo_pose(
        COURT_COCO_DIR,
        COURT_YOLO_POSE_DIR,
        overwrite=FORCE_CONVERSION,
    )
    print("converted data.yaml:", data_yaml)
else:
    print("尚未找到 court dataset。可手動放入 COCO export，或先在 Roboflow 發布新 version，再填入 API 設定並把 USE_ROBOFLOW_DOWNLOAD 改成 True。")


In [ ]:
TRAIN_MODE = "skip"
# 可選值：
# - "skip"   : 不訓練，直接使用課程提供模型
# - "base"   : 以 yolo26n-pose.pt 為初始化權重重新訓練
# - "course" : 以課程提供權重為初始化，繼續微調

if TRAIN_MODE in {"base", "course"}:
    from ultralytics import YOLO

    data_yaml = COURT_YOLO_POSE_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 YOLO pose data.yaml。請先完成 COCO-to-YOLO Pose 轉換。"
        )

    if TRAIN_MODE == "course":
        if not COURT_MODEL_PATH.exists():
            raise FileNotFoundError(
                "找不到課程提供的 court keypoint 權重，無法使用 course 微調模式。"
            )
        init_weights = COURT_MODEL_PATH
    else:
        init_weights = Path("yolo26n-pose.pt")

    model = YOLO(str(init_weights))
    results = model.train(
        data=str(data_yaml),
        epochs=120,
        imgsz=960,
        batch=-1,
        workers=8,
        patience=40,
        fliplr=0.0,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "court_keypoints"),
        name=f"yolo26n_basketball_court_pose_{TRAIN_MODE}",
        plots=True,
    )
    print("init weights:", init_weights)
    print(results)
else:
    print("TRAIN_MODE=skip；本單元直接使用 assets/models/court_keypoints/ 內的權重。")


In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from src.video_utils import display_video_in_notebook

from src.cv_utils import save_json, save_image_rgb, show_image
from src.geometry_utils import render_bev_court
from src.yolo_utils import (
    bev_court_bounds,
    court_keypoints_from_result,
    draw_court_keypoint_records,
    write_court_keypoint_preview_video,
)

BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
INFERENCE_MODEL_PATH = COURT_MODEL_PATH
video_candidates = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if not video_candidates:
    raise FileNotFoundError("找不到參考影片。請確認 assets/raw/reference_videos/ 內至少有一支 mp4。")
video_path = video_candidates[0]

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 60)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"無法讀取 frame 60: {video_path}")
frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
bev = render_bev_court(BEV_SPEC_PATH)

model = YOLO(str(INFERENCE_MODEL_PATH))
result = model.predict(frame, conf=0.15, imgsz=960, verbose=False)[0]
keypoints = court_keypoints_from_result(
    result,
    bev.shape,
    frame_index=60,
    anchor_confidence=0.25,
    court_bounds=bev_court_bounds(BEV_SPEC_PATH),
)

src = np.asarray([kp.image_xy for kp in keypoints], dtype=np.float32)
dst = np.asarray([kp.bev_xy for kp in keypoints], dtype=np.float32)
H = None
if len(src) >= 6:
    H, mask = cv2.findHomography(src, dst, method=cv2.RANSAC, ransacReprojThreshold=5.0)
    if H is not None and mask is not None and int(mask.ravel().sum()) < 6:
        H = None

frame_vis = draw_court_keypoint_records(frame, keypoints)
show_image(frame_vis, "Court keypoint inference on reference frame")

out_image = COURSE_ROOT / "assets" / "results" / "d1_02_court_keypoints.png"
out_json = COURSE_ROOT / "assets" / "results" / "d1_02_court_keypoints.json"
save_image_rgb(out_image, frame_vis)
save_json([record.__dict__ for record in keypoints], out_json)

print("video:", video_path)
print("keypoint count:", len(keypoints))
print("homography estimated:", H is not None)
print("saved frame preview:", out_image)
pd.DataFrame([record.__dict__ for record in keypoints]).head()

preview_video_path = COURSE_ROOT / "assets" / "results" / "d1_02_court_pose_demo.mp4"
preview_video_path, preview_rows = write_court_keypoint_preview_video(
    video_path=video_path,
    model_path=INFERENCE_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=preview_video_path,
    max_frames=120,
    conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=40,
)
preview_json = preview_video_path.with_suffix('.json')

print("saved video demo:", preview_video_path)
print("saved json:", preview_json)
print("frames:", len(preview_rows))
display_video_in_notebook(preview_video_path, loop=True)


## 本單元產出檔案

- `assets/datasets/roboflow_court_coco/`：Roboflow COCO keypoint export（手動放入或 API 下載）。
- `assets/datasets/roboflow_court_yolo_pose/data.yaml` 與對應 images / labels：轉換後的 YOLO pose dataset。
- `assets/results/d1_02_court_keypoints.png`：單張 frame 的 court keypoints 預覽圖。
- `assets/results/d1_02_court_keypoints.json`：單張 frame 的 keypoints 結果。
- `assets/results/d1_02_court_pose_demo.mp4`：court keypoint 預覽影片。
- `assets/results/d1_02_court_pose_demo.json`：預覽影片逐 frame 摘要。
- 若 `TRAIN_MODE != "skip"`，另外會產生 Ultralytics 訓練輸出目錄。
